# FPL Backtest

Replays past seasons with the engine making every decision, using only the data
that existed before each deadline.

There is no engine code in this notebook. It imports the same `fpl_engine.py` the
weekly assistant does, so what gets tuned here is what runs on a Saturday.

**Stages**
- **Projection accuracy** - do the projected points predict real points?
- **Chip timing** - tune `CHIP_TOP_PCT`, the share of remaining weeks a chip will
  accept. Recording a season saves every chip's opportunity table *and* what that
  chip really paid; sweeping the threshold then replays those tables in seconds.
- **Confirmation** - full simulations of the winning settings.

**Caveat printed with every result:** the archive's fixture list is the final one,
so rescheduled double gameweeks are visible here earlier than they were at the
time. That flatters waiting.


In [ ]:
# Setup - fetches the engine. Run once per session.
import os, sys, subprocess

# The one copy of the engine. Point this at your repo once you've pushed it;
# if fpl_engine.py is already sitting next to this notebook it is used as-is,
# so local edits work without pushing anything.
ENGINE_URL = "https://raw.githubusercontent.com/YOUR_USER/YOUR_REPO/main/fpl_engine.py"

if not os.path.exists("fpl_engine.py"):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pulp", "requests"])
    import requests
    r = requests.get(ENGINE_URL, timeout=60); r.raise_for_status()
    open("fpl_engine.py", "w").write(r.text)
    print("engine downloaded")

import fpl_engine as E
import fpl_backtest as B, run_backtest as R
print("engine ready")


## Settings

In [ ]:
SEASONS = ["2023-24", "2024-25", "2025-26"]
SQUADS  = 5          # starting squads per season - 8 chip decisions per run is too few
GRID    = [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]

for s in SEASONS:
    B.download(s)
print("data ready")


## Record

One full season per (season, starting squad), with only the wildcard playable so
the squad trajectory doesn't depend on the decisions being tuned. Roughly 10
minutes per run; they are cached, so a second call skips what is already done.

Run this from a terminal rather than the notebook if you want the progress output:

```
python run_backtest.py record --squads 5 --workers 8
```


In [ ]:
# Or run it here (slower to watch, same result):
# R.cmd_record(R.argparse.Namespace(seasons=",".join(SEASONS), squads=SQUADS,
#                                   gw=38, workers=os.cpu_count()))
records = R.load_records()
print(f"{len(records)} recorded runs:",
      ", ".join(sorted({r["season"] for r in records})) or "none yet")


## Tune the chip thresholds

Replays the saved tables for each candidate `x`. Triple Captain and Bench Boost
never change the squad and Free Hit changes it for one week, so their real
payoffs don't depend on when they were played - which is why this is a sweep over
saved numbers rather than hundreds of re-simulated seasons.


In [ ]:
R.cmd_tune(R.argparse.Namespace(grid=",".join(str(x) for x in GRID)))

## Adopt the winners

Paste the chosen values into `DEFAULTS["CHIP_TOP_PCT"]` in `fpl_engine.py`, then
push. Prefer round numbers over grid-optimal ones - check the standard errors
first; with roughly 8 chip decisions per run, small differences are noise.


In [ ]:
print(json.dumps(E.DEFAULTS["CHIP_TOP_PCT"], indent=2))